In [0]:
%sql
use catalog trueanalytics_data;

In [0]:
import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T 
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

def save_to_csv(df, save_path):
    (df.coalesce(1)
        .write.format('csv')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save csv to ", save_path)

In [0]:
# parameter: par_month
dbutils.widgets.text("par_month", "202510")
par_month = dbutils.widgets.get("par_month")

try:
  par_month = int(par_month)
except ValueError:
  par_month = 0
  raise ValueError("par_month value must be numeric")

if par_month!=0:
  pass
else:
  dbutils.notebook.exit("Aborting as ondition not met. Further tasks will be skipped")

# debug
display(par_month)

In [0]:
# master data
prep_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/proj_3/{par_month}_footfall.parquet'
prep_freq_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/proj_3/{par_month}_flag_freq.parquet'
prep_feature_360 = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/proj_3/{par_month}_360_feature.parquet'
profile_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/profile_chula.csv'
date_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/day_type_apr_june_26.csv'
nantional_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/country_group_chula.csv'
home_region_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/home_region_chula.csv'

# report path
report_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/report/proj_3/report2/{par_month}/'

In [0]:
df = spark.read.parquet(prep_path)
     # raw footfall by mall & customertype (is_bmr, is_non_bmr, is_foriegner)

# df = spark.read.parquet(prep_path)

df_freq = spark.read.parquet(prep_freq_path) # raw freq by mall
df_360 = spark.read.parquet(prep_feature_360)
     # raw profile
df_merge = df.join(F.broadcast(df_freq), ['msisdn','name'], 'left')\
    .join(F.broadcast(df_360.drop('is_bmr','is_non_bmr','is_foriegner','a_country_name','demo_tourist_sim_v1_tourist_bin','roaming_flag')), ['msisdn'], 'inner')
df_merge = df_merge.withColumnRenamed('par_month','month')
display(df_merge.count())

In [0]:
df_date = (spark.read
      .option("header", "true").option("inferSchema", "true")
      .csv(date_path)).select('date','WEEK','day_type_final')

In [0]:
df_merge = df_merge.withColumn('dwelling_time_hr'
                               , F.when(F.col('actual_total_duration_day').between(0, 900), F.lit('less_than_or_equrl_15'))
  .when(F.col('actual_total_duration_day').between(901, 3600), F.lit('more_than_15_<1hr'))
  .when(F.col('actual_total_duration_day').between(3601, 7200), F.lit('1-2'))
  .when(F.col('actual_total_duration_day').between(7201, 10800), F.lit('2-3'))
  .when(F.col('actual_total_duration_day').between(10801, 14400), F.lit('3-4'))
  .when(F.col('actual_total_duration_day').between(14401, 21600), F.lit('4-6'))
  .otherwise(F.lit('>6'))
)

In [0]:
df_intermediate_month = df_merge\
    .join(df_date, [df_merge.par_day == df_date.date], "left")\
    .withColumn(
        "gender_category",
        F.when(F.col("gender") == "F", "gender_female")
         .when(F.col("gender") == "M", "gender_male")
         .otherwise("gender_unidentified"))\
    .withColumn('age_category', 
        F.when(F.col("age_range") == 'unidentified', "age_unidentified")
         .when(F.col("age_range") == '1_12', "age_1_12")
         .when(F.col("age_range") == '13_17', "age_13_17")
         .when(F.col("age_range") == '18_24', "age_18_24")
         .when(F.col("age_range") == '25_34', "age_25_34")
         .when(F.col("age_range") == '35_44', "age_35_44")
         .when(F.col("age_range") == '45_54', "age_45_54")
         .when(F.col("age_range") == '55_59', "age_55_59")
         .when(F.col("age_range") == '60_64', "age_60_64")
         .when(F.col("age_range") >= '>=65', "age_>=65")
         .otherwise("age_unidentified"))\
        .withColumn("visit_frequency", F.concat(F.col("visit_frequency_(day)"), F.lit("_visit_frequency")))\
    
# df_intermediate_month = df_intermediate_month.filter(~((F.col('is_bmr')==0)&(F.col('is_non_bmr')==0)&(F.col('is_foriegner')==0)))
# df_intermediate_month.select('msisdn').distinct().count()

In [0]:
column_core = ['month','latitude','longitude','name','province','district','sub_district']

In [0]:
def pivot_func(df, pivot_col, group_col):
    return (
        df
        .withColumnRenamed('mall', 'name')
        .withColumn(pivot_col, F.lower(F.col(pivot_col)))
        .withColumn(pivot_col, F.regexp_replace(F.col(pivot_col), ' ', '_'))
        .groupBy(group_col)
        .pivot(pivot_col)
        .agg(F.countDistinct("msisdn"))
    )

def group_func(df, group_col, column_name):
    return (
        df\
            .groupBy(group_col)\
            .agg(F.countDistinct("msisdn").alias(column_name))
    )

In [0]:
df_month_bmr = df_intermediate_month.filter((F.col('is_bmr')==1))
df_month_non_bmr = df_intermediate_month.filter((F.col('is_non_bmr')==1))
df_month_foriegner = df_intermediate_month.filter((F.col('is_foriegner')==1))

monthly_bmr_counts = group_func(df_month_bmr, column_core, 'monthly_unique_visitor_cnt')
monthly_non_bmr_counts = group_func(df_month_non_bmr, column_core, 'monthly_unique_visitor_cnt')
monthly_foriegner_counts = group_func(df_month_foriegner, column_core, 'monthly_unique_visitor_cnt')

filter_passer = (F.col('dwelling_time_hr')=='less_than_or_equrl_15')
monthly_passerby_bmr_counts = group_func(df_month_bmr.filter(filter_passer), column_core, 'monthly_unique_passerby_cnt')
monthly_passerby_non_bmr_counts = group_func(df_month_non_bmr.filter(filter_passer), column_core, 'monthly_unique_passerby_cnt')
monthly_passerby_foriegner_counts = group_func(df_month_foriegner.filter(filter_passer), column_core, 'monthly_unique_passerby_cnt')

monthly_non_passerby_bmr_counts = group_func(df_month_bmr.filter(~filter_passer), column_core, 'monthly_unique_non_passerby_cnt')
monthly_non_passerby_non_bmr_counts = group_func(df_month_non_bmr.filter(~filter_passer), column_core, 'monthly_unique_non_passerby_cnt')
monthly_non_passerby_foriegner_counts = group_func(df_month_foriegner.filter(~filter_passer), column_core, 'monthly_unique_non_passerby_cnt')

In [0]:
gender_bmr_counts = pivot_func(df_month_bmr, 'gender_category', column_core)
gender_non_bmr_counts = pivot_func(df_month_non_bmr, 'gender_category', column_core)
gender_foriegner_counts = pivot_func(df_month_foriegner, 'gender_category', column_core)

In [0]:
age_bmr_counts = pivot_func(df_month_bmr, 'age_category', column_core)
age_non_bmr_counts = pivot_func(df_month_non_bmr, 'age_category', column_core)
age_foriegner_counts = pivot_func(df_month_foriegner, 'age_category', column_core)

In [0]:
day_type_bmr_counts = pivot_func(df_month_bmr, 'day_type_final', column_core)
day_type_non_bmr_counts = pivot_func(df_month_non_bmr, 'day_type_final', column_core)
day_type_foriegner_counts = pivot_func(df_month_foriegner, 'day_type_final', column_core)

In [0]:
placetype_bmr_counts = pivot_func(df_month_bmr, 'place_type', column_core)

In [0]:
monthly_pay_bmr_counts = pivot_func(df_month_bmr, 'monthly_pay', column_core)
monthly_pay_non_bmr_counts = pivot_func(df_month_non_bmr, 'monthly_pay', column_core)
monthly_pay_foriegner_counts = pivot_func(df_month_foriegner, 'monthly_pay', column_core)

In [0]:
nationality_foriegner_counts = pivot_func(df_month_foriegner, 'nationality_group', column_core)

In [0]:
def avg_dw_per_day(df, column_name_avg):
    df = df.groupBy('month','name')\
            .agg((F.avg('actual_total_duration_day')/3600).alias(column_name_avg))
    return df

In [0]:
avg_dwl_per_day_bmr = avg_dw_per_day(df_month_bmr, 'average_dwelling_time_per_day')
avg_dwl_per_day_non_bmr = avg_dw_per_day(df_month_non_bmr, 'average_dwelling_time_per_day')
avg_dwl_per_day_foriegner = avg_dw_per_day(df_month_foriegner, 'average_dwelling_time_per_day')

In [0]:
def avg_dw_per_day_multi(df, column_cross, mapping_dict):
    df = df.withColumn(column_cross, F.lower(F.col(column_cross)))
    df = df.withColumn(column_cross, F.regexp_replace(F.col(column_cross), ' ', '_'))
    result_dfs = []
    for value, col_name in mapping_dict.items():
        filtered_df = df.filter(F.col(column_cross) == value)\
            .groupBy('month', 'name')\
            .agg((F.avg('actual_total_duration_day')/3600).alias(col_name))
        result_dfs.append(filtered_df)
    # Join all result_dfs on ['month', 'name']
    from functools import reduce
    final_df = reduce(lambda df1, df2: df1.join(df2, ['month', 'name'], 'outer'), result_dfs)
    return final_df

# Example usage:
day_type_mapping = {
    'holiday_weekend': 'holiday_weekend_dwelling_time',
    'normal_weekend': 'normal_weekend_dwelling_time',
    'special_weekend': 'special_weekend_dwelling_time',
    'normal_weekday': 'normal_weekday_dwelling_time',
    'special_weekday': 'special_weekday_dwelling_time'
}

monthly_pay_mappings = {
    'very_high' : 'average_dwelling_time_per_day_very_high',
    'high' : 'average_dwelling_time_per_day_high',
    'medium' : 'average_dwelling_time_per_day_medium',
    'low' : 'average_dwelling_time_per_day_low',
    'very_low' : 'average_dwelling_time_per_day_very_low',
    'unidentified' : 'average_dwelling_time_per_day_unidentified'
}
nationality_group_mappings = {
    'africa' : 'africa_average_dwelling_time_per_day',
    'asia' : 'asia_average_dwelling_time_per_day',
    'chinese' : 'chinese_average_dwelling_time_per_day',
    'clmv+other_asians' : 'clmv+other_asians_average_dwelling_time_per_day',
    'europe' : 'europe_average_dwelling_time_per_day',
    'high_spending_asians' : 'high_spending_asians_average_dwelling_time_per_day',
    'indian' : 'indian_average_dwelling_time_per_day',
    'middle_east' : 'middle_east_average_dwelling_time_per_day',
    'north_america' : 'north_america_average_dwelling_time_per_day',
    'oceania' : 'oceania_average_dwelling_time_per_day',
    'russian' : 'russian_average_dwelling_time_per_day',
    'south_america' : 'south_america_average_dwelling_time_per_day',
    'u' : 'u_average_dwelling_time_per_day'
}
avg_dwl_x_day_type_bmr = avg_dw_per_day_multi(df_month_bmr, 'day_type_final', day_type_mapping)
avg_dwl_x_day_type_non_bmr = avg_dw_per_day_multi(df_month_non_bmr, 'day_type_final', day_type_mapping)
avg_dwl_x_day_type_foriegner = avg_dw_per_day_multi(df_month_foriegner, 'day_type_final', day_type_mapping)


avg_dwl_x_monthly_pay_bmr = avg_dw_per_day_multi(df_month_bmr, 'monthly_pay', monthly_pay_mappings)
avg_dwl_x_monthly_pay_non_bmr = avg_dw_per_day_multi(df_month_non_bmr, 'monthly_pay', monthly_pay_mappings)
avg_dwl_x_monthly_pay_foriegner = avg_dw_per_day_multi(df_month_foriegner, 'monthly_pay', monthly_pay_mappings)

avg_dwl_x_nationality_foriegner = avg_dw_per_day_multi(df_month_foriegner.withColumnRenamed('mall','name'), 'nationality_group', nationality_group_mappings)

In [0]:
visit_freq_bmr_counts = pivot_func(df_month_bmr, 'visit_frequency', column_core)
visit_freq_non_bmr_counts = pivot_func(df_month_non_bmr, 'visit_frequency', column_core)
visit_freq_foriegner_counts = pivot_func(df_month_foriegner, 'visit_frequency', column_core)

In [0]:
weekly_mall_bmr_counts = pivot_func(df_month_bmr\
    .withColumn('weekly_mall_visitors', F.when(F.col('weekly_mall_visitors') == 1, F.lit('weekly_mall_visitors')).otherwise(F.lit('non_weekly_mall_visitors')))\
    .filter(F.col('weekly_mall_visitors')=='weekly_mall_visitors'), 'weekly_mall_visitors', column_core)

weekly_mall_non_bmr_counts = pivot_func(df_month_non_bmr\
    .withColumn('weekly_mall_visitors', F.when(F.col('weekly_mall_visitors') == 1, F.lit('weekly_mall_visitors')).otherwise(F.lit('non_weekly_mall_visitors')))\
    .filter(F.col('weekly_mall_visitors')=='weekly_mall_visitors'), 'weekly_mall_visitors', column_core)

weekly_mall_foriegner_counts = pivot_func(df_month_foriegner\
    .withColumn('weekly_mall_visitors', F.when(F.col('weekly_mall_visitors') == 1, F.lit('weekly_mall_visitors')).otherwise(F.lit('non_weekly_mall_visitors')))\
    .filter(F.col('weekly_mall_visitors')=='weekly_mall_visitors'), 'weekly_mall_visitors', column_core)

In [0]:
avg_freq_bmr_mnth = df_month_bmr.groupBy('month','name').agg((F.avg('visit_freq_num')).alias('average_frequency_per_month'))
avg_freq_non_bmr_mnth = df_month_non_bmr.groupBy('month','name').agg((F.avg('visit_freq_num')).alias('average_frequency_per_month'))
avg_freq_foriegner_mnth = df_month_foriegner.groupBy('month','name').agg((F.avg('visit_freq_num')).alias('average_frequency_per_month'))

In [0]:
all_bmr = monthly_bmr_counts.join(gender_bmr_counts, column_core, 'left')\
    .join(avg_freq_bmr_mnth, ['month','name'], 'left')\
    .join(age_bmr_counts, column_core, 'left')\
    .join(day_type_bmr_counts, column_core, 'left')\
    .join(placetype_bmr_counts, column_core, 'left')\
    .join(monthly_pay_bmr_counts, column_core, 'left')\
        .join(monthly_passerby_bmr_counts, column_core, 'left')\
            .join(monthly_non_passerby_bmr_counts, column_core, 'left')\
        .join(avg_dwl_per_day_bmr, ['month','name'], 'left')\
            .join(avg_dwl_x_day_type_bmr, ['month','name'], 'left')\
                .join(avg_dwl_x_monthly_pay_bmr, ['month','name'], 'left')\
    .join(visit_freq_bmr_counts, column_core, 'left')\
    .join(weekly_mall_bmr_counts, column_core, 'left')

all_non_bmr = monthly_non_bmr_counts.join(gender_non_bmr_counts, column_core, 'left')\
    .join(avg_freq_non_bmr_mnth, ['month','name'], 'left')\
    .join(age_non_bmr_counts, column_core, 'left')\
    .join(day_type_non_bmr_counts, column_core, 'left')\
    .join(monthly_pay_non_bmr_counts, column_core, 'left')\
        .join(monthly_passerby_non_bmr_counts, column_core, 'left')\
            .join(monthly_non_passerby_non_bmr_counts, column_core, 'left')\
        .join(avg_dwl_per_day_non_bmr, ['month','name'], 'left')\
            .join(avg_dwl_x_day_type_non_bmr, ['month','name'], 'left')\
                .join(avg_dwl_x_monthly_pay_non_bmr, ['month','name'], 'left')\
    .join(visit_freq_non_bmr_counts, column_core, 'left')\
    .join(weekly_mall_non_bmr_counts, column_core, 'left')

all_foriegner = monthly_foriegner_counts.join(gender_foriegner_counts, column_core, 'left')\
    .join(avg_freq_foriegner_mnth, ['month','name'], 'left')\
    .join(age_foriegner_counts, column_core, 'left')\
    .join(day_type_foriegner_counts, column_core, 'left')\
    .join(monthly_pay_foriegner_counts, column_core, 'left')\
        .join(monthly_passerby_foriegner_counts, column_core, 'left')\
            .join(monthly_non_passerby_foriegner_counts, column_core, 'left')\
    .join(nationality_foriegner_counts, column_core, 'left')\
        .join(avg_dwl_per_day_foriegner, ['month','name'], 'left')\
            .join(avg_dwl_x_day_type_foriegner, ['month','name'], 'left')\
                .join(avg_dwl_x_monthly_pay_foriegner, ['month','name'], 'left')\
                .join(avg_dwl_x_nationality_foriegner, ['month','name'], 'left')\
    .join(visit_freq_foriegner_counts, column_core, 'left')\
    .join(weekly_mall_foriegner_counts, column_core, 'left')

In [0]:
save_to_csv(all_bmr, report_path+ f"report2_bmr_{par_month}.csv")
save_to_csv(all_non_bmr, report_path+ f"report2_non_bmr_{par_month}.csv")
save_to_csv(all_foriegner, report_path+ f"report2_foreigner_{par_month}.csv")

In [0]:
for par_month in ['202601','202602','202603', '202604', '202605', '202606']:
    exclude_cols = {"latitude", "longitude"}
    numeric_types = {"integer"}

    volume_path = f"dbfs:/Volumes/int-cu-siampiwat/staging/report/proj_3/report2/{par_month}/"
    path_mask = f"dbfs:/Volumes/int-cu-siampiwat/staging/report/proj_3/mask/report2/{par_month}/"
    files = dbutils.fs.ls(volume_path)
    report1_files = [file.name for file in files]
    for file in report1_files:
        print(file)
        df = spark.read.csv(volume_path+file, header=True, inferSchema=True)
                
        if file.startswith('report2_bmr'):
            df = df.withColumnRenamed('other','home_work_unidentified')\
                .withColumnRenamed('name','mall')

        cols_to_update = [
            field.name for field in df.schema.fields
            if field.dataType.typeName() in numeric_types and field.name not in exclude_cols]

        for col in cols_to_update:
            df = df.withColumn(
                col,
                F.when(F.col(col) == 0, F.lit(0))
                .when(F.col(col) < 25, F.lit(25))
                .otherwise(F.col(col)))

        save_to_csv(df, path_mask+file)